# 301 · Two schema cultures experiment

This notebook goes with the article
[Two schema cultures](https://leo-gan.github.io/GLD.SerializerBenchmark/theory/301/two-schema-cultures/).

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/leo-gan/GLD.SerializerBenchmark/blob/master/docs/theory/notebooks/301/two_schema_cultures.ipynb)

Apache Avro and Protocol Buffers are both “schema-driven,” yet teams operate them with different habits:

- **Field-number culture** (classic Protocol Buffers): numbers on the wire are the stable contract; process and tooling prevent reuse.
- **Resolution culture** (classic Avro): writer and reader schemas can differ; a resolution step fills defaults and projects fields, often with a schema registry.

This notebook uses tiny toys to show the intuition. It is not a full Avro or Protocol Buffers implementation, and it is not a speed contest.

> **A note on numbers:** sizes and timings in these notebooks are only illustrations. For measured library comparisons on this project’s harness, use the suite [Results](https://leo-gan.github.io/GLD.SerializerBenchmark/) pages.


In [ ]:
from __future__ import annotations
from dataclasses import dataclass
from typing import Any, Dict, Tuple


def encode_varint(u: int) -> bytes:
    out = bytearray()
    while u > 0x7F:
        out.append((u & 0x7F) | 0x80)
        u >>= 7
    out.append(u & 0x7F)
    return bytes(out)


def decode_varint(buf: bytes, i: int = 0) -> Tuple[int, int]:
    value = shift = 0
    while True:
        b = buf[i]; i += 1
        value |= (b & 0x7F) << shift
        if b < 0x80:
            return value, i
        shift += 7


def encode_key(fn: int, wt: int) -> bytes:
    return encode_varint((fn << 3) | wt)



## Field-number culture: add field 3; an old reader skips it

Version 2 encodes an `email` value as field number 3.
A version 1 reader only understands fields 1 and 2, so it should keep `id` and `name` and skip the unknown field.

The hard operational rule is: **never reuse field number 3 later for a different meaning.**
In real Protocol Buffers shops, code review and breaking-change detectors enforce that rule more than a runtime dual-schema algorithm does.


In [ ]:
# Writer v2 encodes id=1, name=2, email=3
def encode_pb_v2(id_: int, name: str, email: str) -> bytes:
    out = bytearray()
    out += encode_key(1, 0) + encode_varint(id_)
    nb = name.encode(); out += encode_key(2, 2) + encode_varint(len(nb)) + nb
    eb = email.encode(); out += encode_key(3, 2) + encode_varint(len(eb)) + eb
    return bytes(out)


def decode_pb_v1(buf: bytes) -> dict:
    i = 0
    out = {"id": 0, "name": ""}
    while i < len(buf):
        key, i = decode_varint(buf, i)
        fn, wt = key >> 3, key & 7
        if wt == 0:
            v, i = decode_varint(buf, i)
            if fn == 1:
                out["id"] = v
        elif wt == 2:
            n, i = decode_varint(buf, i)
            payload = buf[i : i + n]; i += n
            if fn == 2:
                out["name"] = payload.decode()
            # unknown (e.g. email=3): skip
        else:
            raise ValueError(wt)
    return out


wire = encode_pb_v2(1, "Ada", "ada@ex.com")
print("v1 reader view:", decode_pb_v1(wire))
print("Culture rule: never reuse field number 3 for a new meaning later.")



## Resolution culture (toy): writer schema, reader schema, and defaults

Here a small function projects data using field **names** and applies defaults when the writer did not send a field.
That is the spirit of Avro-style resolution, not a complete Avro implementation.

In production, a schema registry and compatibility modes such as backward, forward, or full compatibility
are the real control plane. This cell only builds intuition for “two schemas meet at read time.”


In [ ]:
@dataclass
class Field:
    name: str
    typ: str
    default: Any = None


def resolve(writer_schema: list[Field], reader_schema: list[Field], writer_data: Dict[str, Any]) -> Dict[str, Any]:
    """Minimal name-based projection with defaults (Avro-class intuition only)."""
    wset = {f.name: f for f in writer_schema}
    result = {}
    for rf in reader_schema:
        if rf.name in writer_data:
            result[rf.name] = writer_data[rf.name]
        elif rf.default is not None or rf.name not in wset:
            result[rf.name] = rf.default
        else:
            raise KeyError(f"missing required without default: {rf.name}")
    return result


writer_v2 = [Field("id", "int"), Field("name", "string"), Field("email", "string", default="")]
reader_v1 = [Field("id", "int"), Field("name", "string")]
reader_v2 = [Field("id", "int"), Field("name", "string"), Field("email", "string", default="")]

payload_v2 = {"id": 1, "name": "Ada", "email": "ada@ex.com"}
payload_v1 = {"id": 1, "name": "Ada"}

print("writer v2 → reader v1:", resolve(writer_v2, reader_v1, payload_v2))
print("writer v1 → reader v2:", resolve(writer_v1 := writer_v2[:2], reader_v2, payload_v1))
print("Registry would enforce BACKWARD/FORWARD/FULL on schema subjects—not shown here.")



## Decision frame

| If you need… | Lean toward… |
|--------------|--------------|
| Many independent producers and registry checks in continuous integration | Resolution culture (Avro-class) |
| One shared interface definition repo and generated stubs for RPC | Field-number culture (Protocol Buffers-class) |

Neither culture is “more schema-driven.” They differ in **how change is governed**.
Choose the one that matches your operations, and avoid inventing a hybrid without tooling to support it.
